# Setup

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/Code/ptbxl-patient-conditioned-ecg-forecasting"
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import load_config

cfg = load_config(PROJECT_ROOT / "configs/config.yaml")

cfg["data"]["root"] = (
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL"
)

print(cfg["data"])

from data import prepare

cache, manifest = prepare(cfg)

print("Cache:", cache)
print(manifest["splits"])

from pathlib import Path

FIG_DIR = Path("figures")

(FIG_DIR / "patchtst").mkdir(parents=True, exist_ok=True)
(FIG_DIR / "tft").mkdir(parents=True, exist_ok=True)
(FIG_DIR / "comparison").mkdir(parents=True, exist_ok=True)
(FIG_DIR / "leads").mkdir(parents=True, exist_ok=True)


{'root': '/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL', 'sampling_rate': 100, 'history': 700, 'horizon': 300, 'synthetic': False, 'start': 0, 'layout': 'time_first'}
Cache: data/study_v3/ee7d4f585e459326
{'train': {'records': 339, 'patients': 300, 'windows': 339}, 'val': {'records': 81, 'patients': 75, 'windows': 81}}


In [10]:
cfg_large = load_config(
    PROJECT_ROOT / "configs/patchtst_large.yaml"
)

cfg_large["data"]["root"] = (
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL"
)

cfg_large["run_mode"] = "full"

cache_large, manifest_large = prepare(cfg_large)

In [13]:
cfg_tft_large = load_config(
    PROJECT_ROOT / "configs/large.yaml"
)

cfg_tft_large["data"]["root"] = (
    "/media/ehsan/MyFiles3/FraUAS/HIS Project/PTB-XL"
)

cfg_tft_large["run_mode"] = "large"

cache_tft_large, manifest_tft_large = prepare(
    cfg_tft_large
)

print(manifest_tft_large["splits"])

{'train': {'records': 3484, 'patients': 3000, 'windows': 3484}, 'val': {'records': 677, 'patients': 600, 'windows': 677}}


In [1]:
import pandas as pd

def get_best(history_file, name):
    df = pd.read_csv(history_file)

    best = df.loc[
        df["val_patient_macro_mae_mV"].idxmin()
    ]

    return {
        "Model": name,
        "Epoch": int(best["epoch"]),
        "MAE": best["val_patient_macro_mae_mV"],
        "RMSE": best["val_rmse_mV"],
    }

In [2]:
results = pd.DataFrame([
    get_best(
        "../results/primary_7s3s_v3/full/checkpoints/patchtst_P0_F0_full_120e/history.csv",
        "Patch16 P0"
    ),
    get_best(
        "results/primary_7s3s_v3/full/checkpoints/patchtst32_P0_F0_full_60e/history.csv",
        "Patch32 P0"
    ),
    get_best(
        "results/primary_7s3s_v3/full/checkpoints/patchtst_large_P0_F0_full_120e/history.csv",
        "PatchLarge P0"
    ),
    get_best(
        "../results/primary_7s3s_v3/medium/checkpoints/tft_P0_F0_medium_60e/history.csv",
        "TFT Medium P0"
    ),
    get_best(
        "results/primary_7s3s_v3/large/checkpoints/tft_P0_F0_large_80e/history.csv",
        "TFT Large P0"
    ),
    get_best(
        "results/primary_7s3s_v3/large/checkpoints/tft_P1_F0_large_80e/history.csv",
        "TFT Large P1"
    ),
])

results.sort_values("MAE")

,Model,Epoch,MAE,RMSE
4,TFT Large P0,3,0.108185,0.220462
5,TFT Large P1,86,0.108194,0.220468
3,TFT Medium P0,31,0.108556,0.232228
0,Patch16 P0,92,0.113703,0.236500
2,PatchLarge P0,92,0.113703,0.236500
1,Patch32 P0,53,0.115234,0.243206


# Calculating Lead MAE

In [5]:
import torch
import numpy as np
import pandas as pd

from models import build_model
from data import ECGDataset
from preprocessing import feature_arrays

LEAD_NAMES = [
    "I", "II", "III",
    "aVR", "aVL", "aVF",
    "V1", "V2", "V3",
    "V4", "V5", "V6"
]

def compute_lead_mae(
    architecture,
    preprocessing,
    checkpoint_path,
    cfg,
    cache,
):
    arrays, names, details = feature_arrays(
        cfg,
        cache,
        "F0",
        splits=("train", "val")
    )

    dataset = ECGDataset(
        cache,
        cfg,
        "val",
        preprocessing,
        arrays["val"]
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    model = build_model(
        architecture,
        static_dim=0,
        horizon=cfg["data"]["horizon"],
        **cfg["model"]
    )

    model.load_state_dict(checkpoint["model"])
    model.eval()

    lead_errors = []

    with torch.inference_mode():
        for idx in range(len(dataset)):

            sample = dataset[idx]

            history = sample["x"].unsqueeze(0)

            prediction = model(history).squeeze(0)

            target = sample["y"]

            mae_per_lead = (
                prediction - target
            ).abs().mean(dim=0)

            lead_errors.append(
                mae_per_lead.numpy()
            )

    lead_errors = np.asarray(lead_errors)

    mean_lead_mae = lead_errors.mean(axis=0)

    return pd.DataFrame({
        "Lead": LEAD_NAMES,
        "MAE": mean_lead_mae
    })

# PatchTST P0

In [7]:
patch16_leads = compute_lead_mae(
    architecture="patchtst",
    preprocessing="P0",
    checkpoint_path=
    "../results/primary_7s3s_v3/full/checkpoints/"
    "patchtst_P0_F0_full_120e/best.pt",
    cfg=cfg,
    cache=cache,
)

patch16_leads

,Lead,MAE
0,I,0.525251
1,II,0.607377
2,III,0.517576
3,aVR,0.570341
4,aVL,0.500289
5,aVF,0.570719
6,V1,0.503310
7,V2,0.505315
8,V3,0.544292
9,V4,0.575615


# PatchTST Large P0

In [11]:
patchlarge_leads = compute_lead_mae(
    architecture="patchtst",
    preprocessing="P0",
    checkpoint_path=
    "results/primary_7s3s_v3/full/checkpoints/"
    "patchtst_large_P0_F0_full_120e/best.pt",
    cfg=cfg_large,
    cache=cache_large,
)

patchlarge_leads

,Lead,MAE
0,I,0.528230
1,II,0.591155
2,III,0.528173
3,aVR,0.574936
4,aVL,0.510599
5,aVF,0.560394
6,V1,0.455240
7,V2,0.502068
8,V3,0.516779
9,V4,0.531865


# TFT Large P0

In [14]:
tft_large_p0_leads = compute_lead_mae(
    architecture="tft",
    preprocessing="P0",
    checkpoint_path=
    "results/primary_7s3s_v3/large/checkpoints/"
    "tft_P0_F0_large_80e/best.pt",
    cfg=cfg_tft_large,
    cache=cache_tft_large,
)

tft_large_p0_leads

,Lead,MAE
0,I,0.539005
1,II,0.560953
2,III,0.537811
3,aVR,0.554218
4,aVL,0.533921
5,aVF,0.546216
6,V1,0.420766
7,V2,0.454896
8,V3,0.494257
9,V4,0.485256


# TFT Large P1

In [15]:
tft_large_p1_leads = compute_lead_mae(
    architecture="tft",
    preprocessing="P1",
    checkpoint_path=
    "results/primary_7s3s_v3/large/checkpoints/"
    "tft_P1_F0_large_80e/best.pt",
    cfg=cfg_tft_large,
    cache=cache_tft_large,
)

tft_large_p1_leads

,Lead,MAE
0,I,0.540066
1,II,0.561263
2,III,0.538216
3,aVR,0.554744
4,aVL,0.534564
5,aVF,0.546407
6,V1,0.419816
7,V2,0.454829
8,V3,0.494210
9,V4,0.485347


# Saving the results

In [16]:
patch16_leads.to_csv(
    "patch16_leads.csv",
    index=False
)

patchlarge_leads.to_csv(
    "patchlarge_leads.csv",
    index=False
)

tft_large_p0_leads.to_csv(
    "tft_large_p0_leads.csv",
    index=False
)

tft_large_p1_leads.to_csv(
    "tft_large_p1_leads.csv",
    index=False
)

# Best Lead Performance

In [17]:
best_leads = pd.DataFrame({
    "Lead": LEAD_NAMES,
    "PatchTST": patch16_leads["MAE"],
    "TFT Large": tft_large_p0_leads["MAE"]
})

best_leads

,Lead,PatchTST,TFT Large
0,I,0.525251,0.539005
1,II,0.607377,0.560953
2,III,0.517576,0.537811
3,aVR,0.570341,0.554218
4,aVL,0.500289,0.533921
5,aVF,0.570719,0.546216
6,V1,0.503310,0.420766
7,V2,0.505315,0.454896
8,V3,0.544292,0.494257
9,V4,0.575615,0.485256


In [19]:
best_leads.sort_values("TFT Large")

,Lead,PatchTST,TFT Large
6,V1,0.503310,0.420766
7,V2,0.505315,0.454896
11,V6,0.516870,0.483064
9,V4,0.575615,0.485256
10,V5,0.591581,0.487045
8,V3,0.544292,0.494257
4,aVL,0.500289,0.533921
2,III,0.517576,0.537811
0,I,0.525251,0.539005
5,aVF,0.570719,0.546216


In [20]:
best_leads.sort_values("PatchTST")

,Lead,PatchTST,TFT Large
4,aVL,0.500289,0.533921
6,V1,0.503310,0.420766
7,V2,0.505315,0.454896
11,V6,0.516870,0.483064
2,III,0.517576,0.537811
0,I,0.525251,0.539005
8,V3,0.544292,0.494257
3,aVR,0.570341,0.554218
5,aVF,0.570719,0.546216
9,V4,0.575615,0.485256
